# Biohub First Submission

Clean Kaggle/Colab submission notebook. It runs the selected classical baseline config and writes `submission.csv`.

Selected config from local validation:

- percentile_high: `99.8`
- threshold_abs: `0.20`
- gaussian_sigma: `1.2`
- min_distance: `4`
- linking_radius_um: `9.0`

## 1. Imports

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import time

import numpy as np
import pandas as pd

# In Kaggle submission internet is disabled, so dependencies must already be available.
# zarr is optional here; the embedded helper has a small fallback reader for image Zarr v3 chunks.
try:
    import scipy  # noqa: F401
    import skimage  # noqa: F401
except Exception as exc:
    if 'KAGGLE_URL_BASE' in os.environ:
        raise
    print('Installing missing dependencies for Colab development:', repr(exc))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'scipy', 'scikit-image', 'tqdm'])

from tqdm.auto import tqdm

## 2. Embedded Baseline Helper

In [ ]:
baseline_path = Path('biohub_baseline.py')
existing_text = baseline_path.read_text(encoding='utf-8') if baseline_path.exists() else ''
should_write_embedded_helper = (not baseline_path.exists()) or ('SimpleZarrV3ImageArray' not in existing_text)

if should_write_embedded_helper:
    baseline_code = r'''
from pathlib import Path
from typing import Dict, Optional

import numpy as np
import pandas as pd

BASE_DIR = Path('/kaggle/input/competitions/biohub-cell-tracking-during-development')
TRAIN_DIR = BASE_DIR / 'train'
TEST_DIR = BASE_DIR / 'test'
SAMPLE_SUBMISSION_PATH = BASE_DIR / 'sample_submission.csv'

DEBUG_MODE = False
RUN_TEST_SUBMISSION = False

DETECTION_PARAMS = {
    'percentile_low': 1,
    'percentile_high': 99.8,
    'gaussian_sigma': 1.2,
    'min_distance': 4,
    'threshold_abs': 0.20,
    'max_detections_per_frame': None,
}

LINKING_PARAMS = {
    'z_scale': 1.625,
    'y_scale': 0.40625,
    'x_scale': 0.40625,
    'max_distance_um': 9.0,
}


def configure_base_dir(base_dir):
    global BASE_DIR, TRAIN_DIR, TEST_DIR, SAMPLE_SUBMISSION_PATH
    BASE_DIR = Path(base_dir)
    TRAIN_DIR = BASE_DIR / 'train'
    TEST_DIR = BASE_DIR / 'test'
    SAMPLE_SUBMISSION_PATH = BASE_DIR / 'sample_submission.csv'
    return BASE_DIR


def get_zarr():
    try:
        import zarr
        return zarr
    except Exception:
        return None


def _decompress_blosc(data):
    try:
        from numcodecs.blosc import decompress
        return decompress(data)
    except Exception:
        pass
    try:
        import blosc2
        return blosc2.decompress(data)
    except Exception as exc:
        raise ImportError('Need zarr, numcodecs, or blosc2 to read compressed chunks offline.') from exc


class SimpleZarrV3ImageArray:
    def __init__(self, array_path):
        import json
        self.array_path = Path(array_path)
        meta_path = self.array_path / 'zarr.json'
        if not meta_path.exists():
            raise FileNotFoundError(f'Missing Zarr metadata: {meta_path}')
        with open(meta_path, 'r') as f:
            meta = json.load(f)
        self.shape = tuple(int(x) for x in meta['shape'])
        self.dtype = np.dtype(meta.get('data_type', meta.get('dtype', 'uint16')))
        self.chunks = tuple(int(x) for x in meta['chunk_grid']['configuration']['chunk_shape'])
        if self.chunks[0] != 1:
            raise ValueError(f'Fallback reader expects one-timepoint chunks, got {self.chunks}')

    def _chunk_path(self, t):
        return self.array_path / 'c' / str(int(t)) / '0' / '0' / '0'

    def _read_timepoint(self, t):
        chunk_path = self._chunk_path(t)
        if not chunk_path.exists():
            raise FileNotFoundError(f'Missing chunk: {chunk_path}')
        data = chunk_path.read_bytes()
        raw = _decompress_blosc(data)
        arr = np.frombuffer(raw, dtype=self.dtype).reshape(self.chunks)
        return arr[0]

    def __getitem__(self, key):
        if isinstance(key, (int, np.integer)):
            return self._read_timepoint(int(key))
        if isinstance(key, tuple) and len(key) >= 1 and isinstance(key[0], (int, np.integer)):
            arr = self._read_timepoint(int(key[0]))
            return arr[key[1:]]
        raise NotImplementedError('Fallback reader supports img[t] and img[t, ...] only.')


def get_detection_dependencies():
    from scipy.ndimage import gaussian_filter
    from skimage.feature import peak_local_max
    return gaussian_filter, peak_local_max


def get_linking_dependencies():
    from scipy.optimize import linear_sum_assignment
    from scipy.spatial.distance import cdist
    return cdist, linear_sum_assignment


def open_zarr_array(array_path):
    array_path = Path(array_path)
    if not array_path.exists():
        raise FileNotFoundError(f'Missing Zarr array path: {array_path}')
    zarr = get_zarr()
    if zarr is not None:
        return zarr.open(str(array_path), mode='r')
    return SimpleZarrV3ImageArray(array_path)


def open_image_zarr(sample_path, print_info=True):
    sample_path = Path(sample_path)
    img = open_zarr_array(sample_path / '0')
    if print_info:
        print(f'Opened image Zarr: {sample_path}')
        print(f'shape={img.shape}, dtype={img.dtype}, chunks={getattr(img, "chunks", None)}')
    return img


def normalize_volume_percentile(volume_zyx, params):
    volume = volume_zyx.astype(np.float32)
    p_low = np.percentile(volume, params['percentile_low'])
    p_high = np.percentile(volume, params['percentile_high'])
    if p_high <= p_low:
        return np.zeros_like(volume, dtype=np.float32)
    volume = np.clip(volume, p_low, p_high)
    return (volume - p_low) / (p_high - p_low)


def detect_cells_in_timepoint(volume_zyx, t, params):
    gaussian_filter, peak_local_max = get_detection_dependencies()
    volume_norm = normalize_volume_percentile(volume_zyx, params)
    sigma = params.get('gaussian_sigma')
    if sigma and sigma > 0:
        volume_norm = gaussian_filter(volume_norm, sigma=sigma)
    coords = peak_local_max(
        volume_norm,
        min_distance=int(params['min_distance']),
        threshold_abs=float(params['threshold_abs']),
        exclude_border=False,
    )
    if len(coords) == 0:
        return pd.DataFrame(columns=['t', 'z', 'y', 'x', 'score'])
    scores = volume_norm[coords[:, 0], coords[:, 1], coords[:, 2]]
    detections = pd.DataFrame({
        't': int(t),
        'z': coords[:, 0].astype(float),
        'y': coords[:, 1].astype(float),
        'x': coords[:, 2].astype(float),
        'score': scores.astype(float),
    }).sort_values('score', ascending=False)
    max_detections = params.get('max_detections_per_frame')
    if max_detections is not None:
        detections = detections.head(int(max_detections))
    return detections.reset_index(drop=True)


def detect_cells_for_sample(img, sample_id, params, max_timepoints: Optional[int] = None):
    from tqdm.auto import tqdm
    n_timepoints = int(img.shape[0])
    if max_timepoints is not None:
        n_timepoints = min(n_timepoints, int(max_timepoints))
    per_frame = []
    counts = []
    for t in tqdm(range(n_timepoints), desc=f'Detecting {sample_id}'):
        frame_df = detect_cells_in_timepoint(np.asarray(img[t]), t=t, params=params)
        counts.append(len(frame_df))
        per_frame.append(frame_df)
    detections = pd.concat(per_frame, ignore_index=True) if per_frame else pd.DataFrame(columns=['t', 'z', 'y', 'x', 'score'])
    detections.insert(0, 'sample_id', sample_id)
    detections.insert(1, 'node_id', np.arange(1, len(detections) + 1, dtype=int))
    if counts:
        print(f'{sample_id}: detections total={len(detections)}, per-frame min/median/max={np.min(counts)}/{np.median(counts):.1f}/{np.max(counts)}')
    return detections


def voxel_to_physical_um(coords_zyx, params):
    scales = np.array([params['z_scale'], params['y_scale'], params['x_scale']], dtype=float)
    return coords_zyx.astype(float) * scales


def link_detections(detections_df, params):
    cdist, linear_sum_assignment = get_linking_dependencies()
    if detections_df.empty:
        return pd.DataFrame(columns=['sample_id', 'source_id', 'target_id', 'distance_um'])
    sample_id = detections_df['sample_id'].iloc[0]
    edges = []
    unmatched_total = 0
    max_distance_um = float(params['max_distance_um'])
    for t in sorted(detections_df['t'].unique()):
        current_df = detections_df[detections_df['t'] == t].reset_index(drop=True)
        next_df = detections_df[detections_df['t'] == t + 1].reset_index(drop=True)
        if current_df.empty or next_df.empty:
            unmatched_total += len(current_df) + len(next_df)
            continue
        current_um = voxel_to_physical_um(current_df[['z', 'y', 'x']].to_numpy(), params)
        next_um = voxel_to_physical_um(next_df[['z', 'y', 'x']].to_numpy(), params)
        distances = cdist(current_um, next_um)
        row_ind, col_ind = linear_sum_assignment(distances)
        accepted = 0
        for row, col in zip(row_ind, col_ind):
            distance_um = float(distances[row, col])
            if distance_um <= max_distance_um:
                edges.append({
                    'sample_id': sample_id,
                    'source_id': current_df.loc[row, 'node_id'],
                    'target_id': next_df.loc[col, 'node_id'],
                    'distance_um': distance_um,
                })
                accepted += 1
        unmatched_total += (len(current_df) - accepted) + (len(next_df) - accepted)
    edges_df = pd.DataFrame(edges, columns=['sample_id', 'source_id', 'target_id', 'distance_um'])
    if not edges_df.empty:
        print(f'{sample_id}: edges={len(edges_df)}, distance min/median/max={edges_df.distance_um.min():.2f}/{edges_df.distance_um.median():.2f}/{edges_df.distance_um.max():.2f} um, unmatched approx={unmatched_total}')
    else:
        print(f'{sample_id}: no edges accepted, unmatched approx={unmatched_total}')
    return edges_df


def build_submission(detections_all_df, edges_all_df, sample_submission_df, expected_datasets=None):
    required_columns = ['id', 'dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id']
    if list(sample_submission_df.columns) != required_columns:
        raise ValueError(f'Unexpected sample_submission.csv columns: {list(sample_submission_df.columns)}')
    rows = []
    if not detections_all_df.empty:
        node_df = detections_all_df.copy()
        for coord_col in ['t', 'z', 'y', 'x']:
            node_df[coord_col] = node_df[coord_col].round().astype(int)
        node_df['node_id'] = node_df['node_id'].astype(int)
        for row in node_df.itertuples(index=False):
            rows.append({'dataset': row.sample_id, 'row_type': 'node', 'node_id': int(row.node_id), 't': int(row.t), 'z': int(row.z), 'y': int(row.y), 'x': int(row.x), 'source_id': -1, 'target_id': -1})
    if not edges_all_df.empty:
        edge_df = edges_all_df.copy()
        edge_df['source_id'] = edge_df['source_id'].astype(int)
        edge_df['target_id'] = edge_df['target_id'].astype(int)
        for row in edge_df.itertuples(index=False):
            rows.append({'dataset': row.sample_id, 'row_type': 'edge', 'node_id': -1, 't': -1, 'z': -1, 'y': -1, 'x': -1, 'source_id': int(row.source_id), 'target_id': int(row.target_id)})
    submission_df = pd.DataFrame(rows, columns=required_columns[1:])
    submission_df.insert(0, 'id', np.arange(len(submission_df), dtype=int))
    if expected_datasets is not None:
        missing = sorted(set(expected_datasets) - set(submission_df['dataset'].unique()))
        if missing:
            print('Warning: these test datasets have no submission rows:', missing)
    print(f"Submission rows={len(submission_df)}, nodes={(submission_df.row_type == 'node').sum()}, edges={(submission_df.row_type == 'edge').sum()}")
    return submission_df[required_columns]


def save_submission(submission_df, output_path='submission.csv'):
    output_path = Path(output_path)
    submission_df.to_csv(output_path, index=False)
    print(f'Saved submission: {output_path.resolve()}')
    return output_path
'''
    baseline_path.write_text(baseline_code, encoding='utf-8')
    print('Wrote embedded helper:', baseline_path.resolve())
else:
    print('Using existing helper:', baseline_path.resolve())

## 3. Locate Project and Dataset

In [ ]:
PROJECT_CANDIDATES = [
    Path('/kaggle/working'),
    Path('/content/drive/MyDrive/Biohub - Cell Tracking During Development'),
    Path.cwd(),
]

PROJECT_DIR = None
for candidate in PROJECT_CANDIDATES:
    if (candidate / 'biohub_baseline.py').exists():
        PROJECT_DIR = candidate
        break

if PROJECT_DIR is None:
    raise FileNotFoundError('biohub_baseline.py not found. Add it next to this notebook or as a Kaggle dataset/script file.')

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import biohub_baseline as bh


def find_dataset_base():
    candidates = [
        Path('/kaggle/input/competitions/biohub-cell-tracking-during-development'),
        Path('/kaggle/input/biohub-cell-tracking-during-development'),
        Path('/content/drive/MyDrive/Biohub - Cell Tracking During Development/data'),
        Path('/content/drive/MyDrive/Biohub - Cell Tracking During Development/data/biohub-cell-tracking-during-development'),
        PROJECT_DIR / 'data',
        PROJECT_DIR / 'data' / 'biohub-cell-tracking-during-development',
    ]
    for candidate in candidates:
        if (candidate / 'test').exists() and (candidate / 'sample_submission.csv').exists():
            return candidate
    raise FileNotFoundError('Dataset base not found. Expected test/ and sample_submission.csv.')


BASE_DIR = find_dataset_base()
TRAIN_DIR = BASE_DIR / 'train'
TEST_DIR = BASE_DIR / 'test'
SAMPLE_SUBMISSION_PATH = BASE_DIR / 'sample_submission.csv'

bh.configure_base_dir(BASE_DIR)

print('PROJECT_DIR:', PROJECT_DIR)
print('BASE_DIR:', BASE_DIR)
print('test zarr count:', len(list(TEST_DIR.glob('*.zarr'))))

## 3. Selected Parameters

In [ ]:
bh.DETECTION_PARAMS.update({
    'percentile_low': 1,
    'percentile_high': 99.8,
    'gaussian_sigma': 1.2,
    'min_distance': 4,
    'threshold_abs': 0.20,
    'max_detections_per_frame': None,
})

bh.LINKING_PARAMS.update({
    'z_scale': 1.625,
    'y_scale': 0.40625,
    'x_scale': 0.40625,
    'max_distance_um': 9.0,
})

print('DETECTION_PARAMS:', bh.DETECTION_PARAMS)
print('LINKING_PARAMS:', bh.LINKING_PARAMS)

## 4. Build Submission

In [ ]:
sample_submission_df = pd.read_csv(SAMPLE_SUBMISSION_PATH)
test_sample_paths = sorted(TEST_DIR.glob('*.zarr'))
test_sample_ids = [p.stem for p in test_sample_paths]

if not test_sample_paths:
    raise FileNotFoundError(f'No test .zarr samples found under {TEST_DIR}')

all_detections = []
all_edges = []
runtime_rows = []

for sample_path in tqdm(test_sample_paths, desc='test samples'):
    sample_id = sample_path.stem
    img = bh.open_image_zarr(sample_path, print_info=True)

    start = time.time()
    detections = bh.detect_cells_for_sample(img, sample_id, bh.DETECTION_PARAMS)
    detect_seconds = time.time() - start

    start = time.time()
    edges = bh.link_detections(detections, bh.LINKING_PARAMS)
    link_seconds = time.time() - start

    all_detections.append(detections)
    all_edges.append(edges)
    runtime_rows.append({
        'sample_id': sample_id,
        'detections': len(detections),
        'edges': len(edges),
        'detect_seconds': detect_seconds,
        'link_seconds': link_seconds,
    })

detections_all_df = pd.concat(all_detections, ignore_index=True) if all_detections else pd.DataFrame()
edges_all_df = pd.concat(all_edges, ignore_index=True) if all_edges else pd.DataFrame()

submission_df = bh.build_submission(
    detections_all_df,
    edges_all_df,
    sample_submission_df,
    expected_datasets=test_sample_ids,
)

output_path = Path('/kaggle/working/submission.csv') if Path('/kaggle/working').exists() else Path('submission.csv')
bh.save_submission(submission_df, output_path)

runtime_df = pd.DataFrame(runtime_rows)
display(runtime_df)
display(submission_df.head())
print('submission shape:', submission_df.shape)
print('row_type counts:')
print(submission_df['row_type'].value_counts())

## 5. Format Checks

In [ ]:
required_columns = ['id', 'dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id']
assert list(submission_df.columns) == required_columns
assert submission_df['id'].tolist() == list(range(len(submission_df)))
assert set(test_sample_ids).issubset(set(submission_df['dataset'].unique()))
assert set(submission_df['row_type'].unique()).issubset({'node', 'edge'})

node_rows = submission_df[submission_df['row_type'] == 'node']
edge_rows = submission_df[submission_df['row_type'] == 'edge']
assert (node_rows[['source_id', 'target_id']] == -1).all().all()
assert (edge_rows[['node_id', 't', 'z', 'y', 'x']] == -1).all().all()

print('FORMAT CHECK OK')
print('output:', output_path)